# LeetCode 1946 — Largest Number After Mutating Substring bằng Python

Link đề: https://leetcode.com/problems/largest-number-after-mutating-substring/ · Độ khó: **Medium** · Chủ đề: Array, String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maximumNumber(self, num: str, change: List[int]) -> str:
        ...
```

Lộ trình: **thử mọi chuỗi con `[l, r]`, dựng chuỗi mới và so sánh** `O(n³)` → **cố định điểm bắt đầu, mở rộng dần và so sánh thông minh** `O(n²)` → **tham lam một lượt**: bắt đầu ở chữ số đầu tiên **tăng được**, đi tiếp khi **không giảm**, dừng ở chữ số đầu tiên **bị giảm** `O(n)` (lời giải nộp).

## 1. Mục tiêu học

1. So sánh hai số **cùng độ dài** biểu diễn bằng chuỗi: so sánh **từ điển** (lexicographic) = so sánh giá trị — chữ số **đầu tiên khác nhau** quyết định.
2. Lập luận tham lam theo vị trí: tăng chữ số **càng sớm càng tốt** (vị trí cao), và **không bao giờ** để chữ số nào bị giảm.
3. Xử lý ba loại chữ số trong đoạn đột biến: **tăng** (`change[d] > d`), **giữ nguyên** (`=`), **giảm** (`<`) — và vì sao "giữ nguyên" không được làm dừng đoạn.
4. Tránh bẫy `int(...)` với chuỗi 10⁵ chữ số (giới hạn chuyển đổi của Python ≥ 3.11).
5. Kiểm chứng bằng vét cạn mọi chuỗi con trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho chuỗi `num` biểu diễn một số nguyên lớn, và mảng `change` dài 10 ánh xạ chữ số `d` thành `change[d]`.

Bạn được chọn **một** chuỗi con (đoạn liên tiếp) của `num` để **đột biến**: thay mỗi chữ số `num[i]` trong đoạn bằng `change[num[i]]`. (Cũng có thể **không** đột biến.)

Trả về chuỗi biểu diễn số **lớn nhất** có thể.

**Ví dụ 1:** `num = "132"`, `change = [9,8,5,0,3,6,4,2,6,8]` → `"832"` (đột biến `"1"` → `8`).

**Ví dụ 2:** `num = "021"`, `change = [9,4,3,5,7,2,1,9,0,6]` → `"934"` (đột biến cả chuỗi).

**Ví dụ 3:** `num = "5"`, `change = [1,4,7,5,3,2,5,6,9,4]` → `"5"` (không đột biến là tốt nhất).

**Ràng buộc:** `1 ≤ num.length ≤ 10⁵`, `num` gồm chữ số, `change.length == 10`, `0 ≤ change[d] ≤ 9`.

In [1]:
import random
import sys
import time
from typing import List

examples = [
    ("132", [9, 8, 5, 0, 3, 6, 4, 2, 6, 8], "832"),
    ("021", [9, 4, 3, 5, 7, 2, 1, 9, 0, 6], "934"),
    ("5", [1, 4, 7, 5, 3, 2, 5, 6, 9, 4], "5"),
]
for num, ch, want in examples:
    print(repr(num), ch, "->", repr(want))

'132' [9, 8, 5, 0, 3, 6, 4, 2, 6, 8] -> '832'
'021' [9, 4, 3, 5, 7, 2, 1, 9, 0, 6] -> '934'
'5' [1, 4, 7, 5, 3, 2, 5, 6, 9, 4] -> '5'


## 3. Kiến thức nền

### 3.1 So sánh chuỗi chữ số cùng độ dài

Với hai chuỗi **cùng độ dài** chỉ gồm chữ số, `a > b` (so sánh chuỗi) ⇔ số `a` > số `b` — kể cả có số 0 ở đầu.

In [2]:
print("832" > "132", "934" > "021", "0999" < "1000")

True True True


### 3.2 Đột biến một đoạn bằng cắt lát và `join`

In [3]:
def mutate(num, change, l, r):
    return num[:l] + "".join(str(change[int(c)]) for c in num[l:r + 1]) + num[r + 1:]

print(mutate("021", [9, 4, 3, 5, 7, 2, 1, 9, 0, 6], 0, 2))

934


### 3.3 Phân loại chữ số theo `change`

`change[d] > d`: **tăng**; `= d`: **giữ nguyên**; `< d`: **giảm**.

In [4]:
change = [9, 8, 5, 0, 3, 6, 4, 2, 6, 8]
for d in range(10):
    kind = "tăng" if change[d] > d else ("giữ" if change[d] == d else "giảm")
    print(d, "->", change[d], kind)

0 -> 9 tăng
1 -> 8 tăng
2 -> 5 tăng
3 -> 0 giảm
4 -> 3 giảm
5 -> 6 tăng
6 -> 4 giảm
7 -> 2 giảm
8 -> 6 giảm
9 -> 8 giảm


### 3.4 Giới hạn chuyển `str` → `int` của Python ≥ 3.11

Để tránh tấn công từ chối dịch vụ, `int("…")` với chuỗi quá **4300** chữ số mặc định sẽ ném `ValueError`. Bài có `num` dài tới `10⁵` ⇒ **đừng** chuyển sang `int` để so sánh.

In [5]:
print("giới hạn hiện tại:", sys.get_int_max_str_digits() if hasattr(sys, "get_int_max_str_digits") else "không có")
try:
    int("9" * 5000)
    print("chuyển đổi được")
except ValueError as e:
    print("ValueError:", str(e)[:70])

giới hạn hiện tại: 4300
ValueError: Exceeds the limit (4300 digits) for integer string conversion: value h


### 3.5 `list` của ký tự rồi `"".join` — sửa nhiều vị trí trong `O(n)`

Chuỗi bất biến; sửa từng ký tự bằng cắt lát là `O(n)` mỗi lần. Chuyển sang list, sửa tại chỗ, `join` một lần.

In [6]:
s = list("021")
s[0] = "9"
print("".join(s))

921


## 4. Ý tưởng nền tảng / trực giác

Số lớn hơn được quyết định bởi chữ số **đầu tiên khác nhau** (từ trái sang). Vậy:

1. Tìm vị trí **đầu tiên** `i` mà `change[num[i]] > num[i]` — bắt đầu đột biến ở đây làm chữ số ở vị trí **cao nhất có thể** tăng lên. (Bắt đầu sớm hơn chỉ đụng vào chữ số giữ nguyên hoặc giảm — vô ích hoặc có hại.)
2. Từ `i` đi tiếp sang phải **chừng nào** `change[d] ≥ d` — chữ số tăng thì có lợi, giữ nguyên thì vô hại (và cho phép đi tới các chữ số tăng phía sau).
3. Dừng ngay ở chữ số đầu tiên có `change[d] < d` — đột biến nó làm số nhỏ đi, mà đoạn phải **liên tiếp** nên không thể "nhảy qua".

Ví dụ 2 `"021"`: `0 → 9` (tăng, bắt đầu), `2 → 3` (tăng), `1 → 4` (tăng) ⇒ `"934"`.

Ví dụ 1 `"132"`: `1 → 8` (tăng, bắt đầu), `3 → 0` (giảm) ⇒ dừng ⇒ `"832"`.

## 5. Cách 1 — Brute Force / baseline: thử mọi chuỗi con

In [7]:
def maximum_brute(num, change):
    best = num                                       # không đột biến
    n = len(num)
    for l in range(n):
        for r in range(l, n):
            cand = mutate(num, change, l, r)
            if cand > best:
                best = cand
    return best

for num, ch, want in examples:
    print(repr(maximum_brute(num, ch)), "| kỳ vọng", repr(want))

'832' | kỳ vọng '832'
'934' | kỳ vọng '934'
'5' | kỳ vọng '5'


**Độ phức tạp:** `O(n²)` đoạn × `O(n)` dựng và so ⇒ `O(n³)`. `n = 10⁵` là bất khả. **Nút thắt:** đa số đoạn hiển nhiên tệ (chứa chữ số bị giảm ở vị trí cao, hoặc bắt đầu muộn hơn cần thiết).

In [8]:
num_rand = "".join(random.Random(0).choice("0123456789") for _ in range(120))
t = time.perf_counter(); maximum_brute(num_rand, examples[0][1]); dt = time.perf_counter() - t
print(f"n=120: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * (10**5 / 120) ** 3 / 3600 / 24 / 365:.0f} năm")

n=120: 0.03s -> n=10^5 ước tính ≈ 0 năm


## 6. Ý tưởng thuật toán / bước đột phá

**Quy tắc tham lam (ba "nếu … thì …"):**
- *Nếu* chưa bắt đầu và `change[d] > d` *thì* **bắt đầu** đột biến tại đây.
- *Nếu* đã bắt đầu và `change[d] ≥ d` *thì* **tiếp tục** (đột biến chữ số này).
- *Nếu* đã bắt đầu và `change[d] < d` *thì* **dừng hẳn**.

Nếu không có chữ số nào tăng được ⇒ trả `num` nguyên vẹn.

**Vì sao bắt đầu ở chữ số tăng đầu tiên?** Gọi vị trí đó là `i`. Mọi phương án tốt phải giữ nguyên `num[0..i−1]` (không thể làm chúng lớn hơn; nếu làm nhỏ hơn thì thua ngay) và nên tăng chữ số `i` (nếu không tăng `i` thì chữ số đầu tiên khác `num` nằm sau `i` — kém hơn phương án tăng `i`).

**Invariant:** đoạn đột biến là `[i, j)`, với `j` là vị trí đầu tiên `> i` có `change < d` (hoặc `n`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cố định điểm bắt đầu, mở rộng dần (`O(n²)`)

Với mỗi `l`, mở rộng `r` và duy trì **list** chữ số đang đột biến; so sánh chuỗi ứng viên với tốt nhất. Giảm được việc dựng lại phần đầu, nhưng vẫn so sánh `O(n)` mỗi ứng viên — ta tối ưu thêm: từ một `l`, ứng viên tốt nhất là đoạn dài nhất **không chứa chữ số giảm** (mở rộng thêm chữ số tăng/giữ không bao giờ làm tệ đi).

In [9]:
def maximum_per_start(num, change):
    n = len(num); best = num
    for l in range(n):
        r = l
        while r < n and change[int(num[r])] >= int(num[r]):
            r += 1
        if r > l:
            cand = mutate(num, change, l, r - 1)
            if cand > best:
                best = cand
    return best

for num, ch, want in examples:
    print(repr(maximum_per_start(num, ch)), repr(want))

'832' '832'
'934' '934'
'5' '5'


**Độ phức tạp:** `O(n)` điểm bắt đầu × `O(n)` mở rộng/so sánh ⇒ `O(n²)`. Nút thắt: thử **mọi** điểm bắt đầu, trong khi chỉ điểm "tăng đầu tiên" là đáng xét.

### 7.2 Cách 3 — Tham lam một lượt (`O(n)`) — lời giải chuẩn

In [10]:
def maximum_greedy_demo(num, change):
    s = list(num); started = False
    for i, c in enumerate(s):
        d = int(c); m = change[d]
        if not started:
            if m > d:
                started = True; s[i] = str(m)
                print(f"i={i}: {d} -> {m} (tăng) — BẮT ĐẦU")
            else:
                print(f"i={i}: {d} -> {m} ({'giữ' if m == d else 'giảm'}) — chưa bắt đầu, bỏ qua")
        else:
            if m >= d:
                s[i] = str(m); print(f"i={i}: {d} -> {m} ({'tăng' if m > d else 'giữ'}) — tiếp tục")
            else:
                print(f"i={i}: {d} -> {m} (giảm) — DỪNG"); break
    return "".join(s)

print("=>", maximum_greedy_demo("021", examples[1][1]))

i=0: 0 -> 9 (tăng) — BẮT ĐẦU
i=1: 2 -> 3 (tăng) — tiếp tục
i=2: 1 -> 4 (tăng) — tiếp tục
=> 934


In [11]:
class SolutionGreedy:
    def maximumNumber(self, num: str, change: List[int]) -> str:
        s = list(num)
        i, n = 0, len(s)
        while i < n and change[int(s[i])] <= int(s[i]):      # tìm chữ số tăng được đầu tiên
            i += 1
        while i < n and change[int(s[i])] >= int(s[i]):      # đột biến tới khi gặp chữ số bị giảm
            s[i] = str(change[int(s[i])])
            i += 1
        return "".join(s)

for num, ch, want in examples:
    print(SolutionGreedy().maximumNumber(num, ch) == want)

True
True
True


**Độ phức tạp:** một lượt `O(n)` thời gian, `O(n)` bộ nhớ cho list (bắt buộc để trả chuỗi mới).

### 7.3 Cách 4 — Cùng tham lam, dùng bảng chuỗi ánh xạ và cắt lát một lần

Xác định `[l, r)` trước rồi dựng kết quả bằng `num[:l] + mapped + num[r:]` — dùng `str.translate` với bảng ánh xạ chữ số, chạy bằng C.

In [12]:
class SolutionTranslate:
    def maximumNumber(self, num: str, change: List[int]) -> str:
        n = len(num)
        l = 0
        while l < n and change[ord(num[l]) - 48] <= ord(num[l]) - 48:
            l += 1
        if l == n:
            return num
        r = l
        while r < n and change[ord(num[r]) - 48] >= ord(num[r]) - 48:
            r += 1
        table = str.maketrans("0123456789", "".join(map(str, change)))
        return num[:l] + num[l:r].translate(table) + num[r:]

for num, ch, want in examples:
    print(SolutionTranslate().maximumNumber(num, ch) == want)

True
True
True


`str.maketrans` + `translate` ánh xạ từng ký tự bằng mã C — nhanh hơn vòng lặp Python khi đoạn dài.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 1 (`change[1] = 8`, `change[3] = 0`, `change[2] = 5`):

| i | chữ số | `change` | loại | trạng thái | chuỗi |
|---|---|---|---|---|---|
| 0 | 1 | 8 | tăng | bắt đầu | `832` |
| 1 | 3 | 0 | giảm | **dừng** | `832` |

Kết quả `"832"`. Trace một ví dụ tự dựng có chữ số "giữ" ở giữa đoạn (`num = "121"`, `change[1] = 9`, `change[2] = 2`):

In [13]:
ch = list(range(10)); ch[1] = 9                       # 1 -> 9, mọi chữ khác giữ nguyên
print("=>", maximum_greedy_demo("121", ch), "| brute:", maximum_brute("121", ch))

i=0: 1 -> 9 (tăng) — BẮT ĐẦU
i=1: 2 -> 2 (giữ) — tiếp tục
i=2: 1 -> 9 (tăng) — tiếp tục
=> 929 | brute: 929


In [14]:
ch2 = list(range(10)); ch2[2] = 9; ch2[3] = 0          # 1 giữ, 3 giảm, 2 tăng
print("=>", maximum_greedy_demo("132", ch2), "| brute:", maximum_brute("132", ch2))

i=0: 1 -> 1 (giữ) — chưa bắt đầu, bỏ qua
i=1: 3 -> 0 (giảm) — chưa bắt đầu, bỏ qua
i=2: 2 -> 9 (tăng) — BẮT ĐẦU
=> 139 | brute: 139


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def maximumNumber(self, num: str, change: List[int]) -> str:
        s = list(num)
        n, i = len(s), 0
        while i < n and change[int(s[i])] <= int(s[i]):      # 1) bỏ qua tới chữ số TĂNG được đầu tiên
            i += 1
        while i < n and change[int(s[i])] >= int(s[i]):      # 2) đột biến khi còn không giảm
            s[i] = str(change[int(s[i])])
            i += 1
        return "".join(s)                                     # 3) dừng ở chữ số bị giảm đầu tiên

for num, ch, want in examples:
    assert Solution().maximumNumber(num, ch) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `i` = vị trí đầu tiên có `change[num[i]] > num[i]`. Nếu không có `i` thì mọi đột biến chỉ giữ nguyên hoặc làm giảm chữ số đầu tiên bị đổi ⇒ `num` là tối ưu (thuật toán trả `num`).

Giả sử có `i`, và `j` = vị trí đầu tiên `> i` có `change < d` (hoặc `n`). Thuật toán trả `G` = đột biến `[i, j)`.

**So với một phương án `[l, r]` bất kỳ:**
- Nếu `l < i`: các vị trí `l..i−1` (nếu được đột biến) chỉ **giữ nguyên hoặc giảm**. Nếu có vị trí bị giảm, gọi `p < i` là vị trí giảm đầu tiên: phương án và `G` trùng nhau trước `p`, tại `p` phương án **nhỏ hơn** ⇒ `G` thắng. Nếu không có vị trí giảm thì phương án trùng `num` (và `G`) trên `0..i−1`; tại `i` nó hoặc tăng giống `G` (khi `r ≥ i`, khi đó quy về trường hợp `l = i` bên dưới), hoặc không đổi (khi `r < i`) ⇒ `G` lớn hơn tại `i`.
- Nếu `l > i`: phương án giữ nguyên `num[i]` còn `G` tăng nó, mọi vị trí trước bằng nhau ⇒ `G` thắng.
- Nếu `l = i`: hai bên giống nhau trên `[i, min(r+1, j))`. Nếu `r + 1 < j`, `G` đột biến thêm các vị trí `≥` (không giảm) ⇒ `G ≥`. Nếu `r + 1 > j`, phương án đột biến vị trí `j` (bị giảm) ⇒ nhỏ hơn `G` tại `j`.

⇒ `G` lớn nhất. Mục 12 đối chiếu thêm với vét cạn.

## 11. Phân tích Time / Space

Gọi `n = len(num) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi chuỗi con | `O(n³)` | `O(n)` |
| 2. Mỗi điểm bắt đầu | `O(n²)` | `O(n)` |
| 3. Tham lam | `O(n)` | `O(n)` |
| 4. Tham lam + translate | `O(n)` (hằng số nhỏ) | `O(n)` |

In [16]:
rng = random.Random(1946)
big = "".join(rng.choice("0123456789") for _ in range(10**5))
ch_up = [min(9, d + 1) for d in range(10)]                    # mọi chữ tăng/giữ: đột biến gần hết chuỗi
ch_rand = [rng.randint(0, 9) for _ in range(10)]
for label, ch in [("toàn tăng", ch_up), ("ngẫu nhiên", ch_rand)]:
    res = []
    for name, sol in [("Tham lam/Nộp", Solution()), ("Translate", SolutionTranslate())]:
        t = time.perf_counter(); r = sol.maximumNumber(big, ch); res.append(r)
        print(f"[{label}] {name}: {1000*(time.perf_counter()-t):.1f} ms")
    assert len(set(res)) == 1
t = time.perf_counter(); maximum_per_start(big[:1500], ch_rand); dt = time.perf_counter() - t
print(f"Cách 2 với n=1500: {dt:.2f}s")

[toàn tăng] Tham lam/Nộp: 18.5 ms
[toàn tăng] Translate: 6.0 ms
[ngẫu nhiên] Tham lam/Nộp: 0.7 ms
[ngẫu nhiên] Translate: 0.0 ms
Cách 2 với n=1500: 0.00s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [17]:
sols = [Solution(), SolutionGreedy(), SolutionTranslate()]
ident = list(range(10))
edge = examples + [
    ("9", ident, "9"),                            # không đổi gì
    ("0", [1] + ident[1:], "1"),
    ("121", [0, 9] + ident[2:], "929"),           # chữ số "giữ" ở giữa không làm dừng
    ("132", [0, 1, 9, 0] + ident[4:], "139"),     # bắt đầu ở chữ số "giữ" sẽ sai
    ("1312", [0, 9, 2, 0] + ident[4:], "9312"),   # chỉ MỘT đoạn
    ("999", [0] * 10, "999"),                     # mọi chữ bị giảm: không đột biến
    ("000", [9] * 10, "999"),
]
for num, ch, want in edge:
    assert maximum_brute(num, ch) == want, (num, ch, maximum_brute(num, ch))
    for s in sols:
        assert s.maximumNumber(num, ch) == want, (num, ch, s)
    assert maximum_per_start(num, ch) == want
    print("OK", repr(num), "->", repr(want))

OK '132' -> '832'
OK '021' -> '934'
OK '5' -> '5'
OK '9' -> '9'
OK '0' -> '1'
OK '121' -> '929'
OK '132' -> '139'
OK '1312' -> '9312'
OK '999' -> '999'
OK '000' -> '999'


### 12.2 Ngẫu nhiên có seed: so với vét cạn

In [18]:
rng = random.Random(42)
for _ in range(1500):
    num = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 12)))
    ch = [rng.randint(0, 9) for _ in range(10)]
    want = maximum_brute(num, ch)
    for s in sols:
        assert s.maximumNumber(num, ch) == want, (num, ch)
    assert maximum_per_start(num, ch) == want
print("1500 bộ ngẫu nhiên: khớp vét cạn")

1500 bộ ngẫu nhiên: khớp vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Bắt đầu đột biến ở chữ số "giữ nguyên" (`change[d] >= d`)

In [19]:
def wrong_start_equal(num, change):
    s = list(num); i, n = 0, len(s)
    while i < n and change[int(s[i])] < int(s[i]):          # SAI: dừng cả ở chữ số "giữ"
        i += 1
    while i < n and change[int(s[i])] >= int(s[i]):
        s[i] = str(change[int(s[i])]); i += 1
    return "".join(s)

num, ch = "132", [0, 1, 9, 0] + list(range(4, 10))
print("sai:", wrong_start_equal(num, ch), "| đúng:", Solution().maximumNumber(num, ch))

sai: 132 | đúng: 139


**Vì sao sai:** đoạn bắt đầu ở `1` (giữ nguyên), gặp ngay `3 → 0` (giảm) nên dừng — chưa kịp tới chữ số `2 → 9`. Phải bắt đầu ở chữ số **tăng thật sự** đầu tiên.

### Lỗi 2 — Dừng ở chữ số "giữ nguyên" (`change[d] > d` cho cả vòng tiếp tục)

In [20]:
def wrong_stop_equal(num, change):
    s = list(num); i, n = 0, len(s)
    while i < n and change[int(s[i])] <= int(s[i]):
        i += 1
    while i < n and change[int(s[i])] > int(s[i]):          # SAI: dừng cả khi bằng
        s[i] = str(change[int(s[i])]); i += 1
    return "".join(s)

num, ch = "121", [0, 9] + list(range(2, 10))
print("sai:", wrong_stop_equal(num, ch), "| đúng:", Solution().maximumNumber(num, ch))

sai: 921 | đúng: 929


**Vì sao sai:** chữ số `2` giữ nguyên — đột biến nó **vô hại** và cho phép đoạn kéo tới `1 → 9` phía sau. Dừng sớm cho `"921"` thay vì `"929"`.

### Lỗi 3 — Đột biến **mọi** vị trí tăng được (nhiều đoạn rời nhau)

In [21]:
def wrong_all_positions(num, change):
    return "".join(str(max(int(c), change[int(c)])) for c in num)    # SAI: không còn là MỘT đoạn

num, ch = "1312", [0, 9, 2, 0] + list(range(4, 10))
print("sai:", wrong_all_positions(num, ch), "| đúng:", Solution().maximumNumber(num, ch))

sai: 9392 | đúng: 9312


**Vì sao sai:** đề cho phép đột biến **một** chuỗi con liên tiếp. `"1312"`: tăng `1` đầu, gặp `3 → 0` thì phải dừng; không được "bỏ qua" `3` để tăng `1` thứ hai.

### Lỗi 4 — So sánh bằng `int(...)` với số rất dài

In [22]:
def wrong_int_compare(num, change):
    best = num
    for l in range(len(num)):
        cand = mutate(num, change, l, l)
        if int(cand) > int(best):                            # chuyển cả chuỗi sang int
            best = cand
    return best

long_num = "1" * 6000
try:
    wrong_int_compare(long_num, [0, 9] + list(range(2, 10)))
    print("không lỗi (giới hạn chuyển đổi đã được tắt?)")
except ValueError as e:
    print("ValueError:", str(e)[:60], "| so sánh chuỗi thì không sao:", Solution().maximumNumber(long_num, [0, 9] + list(range(2, 10)))[:10] + "…")

ValueError: Exceeds the limit (4300 digits) for integer string conversio | so sánh chuỗi thì không sao: 9999999999…


**Vì sao sai:** từ Python 3.11, chuyển chuỗi > 4300 chữ số sang `int` bị chặn (`ValueError`); kể cả không bị chặn thì chuyển đổi là `O(n²)`. Hai chuỗi cùng độ dài ⇒ **so sánh chuỗi** là đủ và nhanh.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Làm số lớn nhất / nhỏ nhất bằng một thao tác trên đoạn"** ⇒ tham lam từ trái: vị trí đầu tiên có lợi là nơi bắt đầu; kéo dài khi không hại; dừng khi có hại.
- Phân biệt rõ ba loại tác động: **có lợi / trung tính / có hại** — trung tính thường **không** là lý do để dừng.
- So sánh số dạng chuỗi cùng độ dài: dùng so sánh chuỗi.
- Sửa nhiều ký tự: list + `join`, hoặc `str.translate` cho ánh xạ ký tự.
- Bài tương tự: LeetCode 670 *Maximum Swap*, 1323 *Maximum 69 Number*, 2566.

## 15. Biến thể của bài

1. **Số nhỏ nhất** sau khi đột biến một đoạn (Bài tập 5) — đổi vai trò "tăng/giảm", cẩn thận số 0 đầu.
2. **Được đột biến tối đa `k` đoạn** ⇒ tham lam: lấy `k` đoạn "có lợi" đầu tiên… nhưng giữa các đoạn phải có chữ số bị giảm.
3. **Ánh xạ khác nhau cho từng vị trí** ⇒ cùng tham lam với `change_i`.
4. **Bắt buộc đột biến** (đoạn khác rỗng) ⇒ nếu không có vị trí tăng, chọn đoạn "ít hại nhất" (ưu tiên chữ số giữ nguyên, hoặc chữ số giảm ở vị trí thấp nhất).
5. **LeetCode 670 — Maximum Swap:** một lần hoán đổi hai chữ số.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi chuỗi con | thử hết, so chuỗi | `O(n³)` | `O(n)` | kiểm chứng |
| 2. Mỗi điểm bắt đầu | đoạn dài nhất không giảm | `O(n²)` | `O(n)` | bước trung gian |
| 3. Tham lam | bắt đầu ở chữ tăng đầu tiên, dừng ở chữ giảm | `O(n)` | `O(n)` | **nộp bài** |
| 4. Tham lam + translate | như 3, ánh xạ bằng C | `O(n)` | `O(n)` | chuỗi rất dài |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `("5", change với change[5] = 6)`, `("19", change = [0..9] nhưng change[1]=1, change[9]=0)`?

**Bài 2 (tính tay):** Trace tham lam cho Ví dụ 2.

**Bài 3 (điền TODO):** Viết `best_range(num, change)` trả về `(l, r)` (nửa mở) của đoạn tham lam chọn, hoặc `None`.

**Bài 4 (tự viết lại):** Viết lời giải **không** tạo list — chỉ tìm `l, r` rồi ghép chuỗi một lần (giống Cách 4 nhưng không dùng `translate`).

**Bài 5 (mini challenge):** Viết `minimum_number(num, change)` — số **nhỏ nhất** sau khi đột biến (tối đa) một chuỗi con.

In [23]:
def best_range(num, change):
    # TODO
    pass

def maximum_no_list(num, change):
    # TODO
    pass

def minimum_number(num, change):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `5 → 6` tăng | `"6"` | `"19"`: 1 giữ, 9 giảm ⇒ không đột biến, `"19"` |
| 2 | `0 → 9` tăng: bắt đầu | `2 → 3`, `1 → 4` đều tăng | `"934"` |
| 3 | Vòng `while` thứ nhất cho `l` | Vòng thứ hai cho `r` | `l == n` ⇒ `None` |
| 4 | `num[:l] + mapped + num[r:]` | `mapped = "".join(str(change[int(c)]) for c in num[l:r])` | Một lần ghép |
| 5 | Đối xứng: bắt đầu ở chữ **giảm** được đầu tiên | Tiếp tục khi `change[d] ≤ d` | Dừng khi `change[d] > d` |

## 19. Đáp án tham khảo

In [24]:
c1 = list(range(10)); c1[5] = 6
c2 = list(range(10)); c2[9] = 0
r1 = [Solution().maximumNumber("5", c1), Solution().maximumNumber("19", c2)]
print("Bài 1:", r1); assert r1 == ["6", "19"]
print("Bài 2:"); assert maximum_greedy_demo("021", examples[1][1]) == "934"

Bài 1: ['6', '19']
Bài 2:
i=0: 0 -> 9 (tăng) — BẮT ĐẦU
i=1: 2 -> 3 (tăng) — tiếp tục
i=2: 1 -> 4 (tăng) — tiếp tục


In [25]:
def best_range(num, change):
    n = len(num); l = 0
    while l < n and change[int(num[l])] <= int(num[l]): l += 1
    if l == n: return None
    r = l
    while r < n and change[int(num[r])] >= int(num[r]): r += 1
    return l, r

def maximum_no_list(num, change):
    rng_ = best_range(num, change)
    if rng_ is None: return num
    l, r = rng_
    return num[:l] + "".join(str(change[int(c)]) for c in num[l:r]) + num[r:]

rng = random.Random(3)
for _ in range(800):
    num = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 12)))
    ch = [rng.randint(0, 9) for _ in range(10)]
    assert maximum_no_list(num, ch) == maximum_brute(num, ch)
print("Bài 3, 4: OK", best_range("132", examples[0][1]), best_range("021", examples[1][1]), best_range("5", examples[2][1]))

Bài 3, 4: OK (0, 1) (0, 3) None


In [26]:
def minimum_number(num, change):
    s = list(num); i, n = 0, len(s)
    while i < n and change[int(s[i])] >= int(s[i]): i += 1
    while i < n and change[int(s[i])] <= int(s[i]):
        s[i] = str(change[int(s[i])]); i += 1
    return "".join(s)

def minimum_brute(num, change):
    best = num
    for l in range(len(num)):
        for r in range(l, len(num)):
            best = min(best, mutate(num, change, l, r))
    return best

rng = random.Random(9)
for _ in range(1000):
    num = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 10)))
    ch = [rng.randint(0, 9) for _ in range(10)]
    assert minimum_number(num, ch) == minimum_brute(num, ch), (num, ch)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", minimum_number("832", examples[0][1]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 602


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
s = list(num); i, n = 0, len(s)
while i < n and change[int(s[i])] <= int(s[i]): i += 1   # tìm chữ TĂNG đầu tiên
while i < n and change[int(s[i])] >= int(s[i]):           # tăng hoặc giữ: tiếp tục
    s[i] = str(change[int(s[i])]); i += 1
return "".join(s)                                          # gặp chữ GIẢM: dừng
```

**Checklist:**
- [ ] Bắt đầu ở chữ số **tăng thật sự** (`>`), không phải `>=`?
- [ ] Tiếp tục qua chữ số **giữ nguyên** (`>=`)?
- [ ] Chỉ **một** đoạn liên tiếp, dừng hẳn ở chữ số bị giảm?
- [ ] Không có chữ số tăng ⇒ trả `num`?
- [ ] So sánh/biến đổi trên chuỗi, không dùng `int(num)`.